# Preference Tuning with DPO — Code Companion (Unsloth + Llama 3.1 (8B))

This notebook accompanies **Topic: Preference Tuning with DPO**.

We continue directly from the LoRA & QLoRA notebook: starting with our SFT'd Llama 3.1
(8B) checkpoint, we now teach it *which* of two responses people actually prefer, using
**Direct Preference Optimization (DPO)** via **Unsloth**.

> **Important:** like the LoRA/QLoRA notebook, this needs a GPU and internet access. It
> will not run inside a CPU-only, offline sandbox. Every cell is complete, runnable code
> — open this in Colab (GPU runtime) and run top to bottom, ideally right after running
> the LoRA & QLoRA notebook so `./lora_model` exists.

**What you'll do:**
1. Understand the shape of preference data — hands-on, with a tiny toy example.
2. Build a small preference dataset.
3. Load our already-SFT'd model with Unsloth and patch in DPO support.
4. Configure and run `DPOTrainer` from TRL.
5. Compare the model's outputs before and after DPO.
6. Inspect the DPO loss curve and the reward margin between chosen and rejected responses.

## 1. The Shape of Preference Data, Concretely

Every DPO training example is a **(prompt, chosen, rejected)** triplet: one prompt, two
candidate responses, and a label for which one is preferred. Let's look at a few by hand
before building a full dataset.

In [1]:
preference_examples = [
    {
        "prompt": "Explain photosynthesis to a 10-year-old.",
        "chosen": "Plants use sunlight, water, and air to make their own food -- "
                   "like a tiny kitchen powered by the sun!",
        "rejected": "Photosynthesis is the process by which chlorophyll-containing "
                     "organisms convert electromagnetic radiation into chemical energy "
                     "via a series of light-dependent and light-independent reactions.",
    },
    {
        "prompt": "Write a short, friendly out-of-office reply.",
        "chosen": "Thanks for your email! I'm out of office until Monday and will "
                   "reply as soon as I'm back. For anything urgent, please contact "
                   "my colleague Sam.",
        "rejected": "I am currently unavailable.",
    },
]

for i, ex in enumerate(preference_examples):
    print(f"Example {i}:")
    print(f"  prompt:   {ex['prompt']!r}")
    print(f"  chosen:   {ex['chosen'][:70]}...")
    print(f"  rejected: {ex['rejected'][:70]}...")
    print()

Example 0:
  prompt:   'Explain photosynthesis to a 10-year-old.'
  chosen:   Plants use sunlight, water, and air to make their own food -- like a t...
  rejected: Photosynthesis is the process by which chlorophyll-containing organism...

Example 1:
  prompt:   'Write a short, friendly out-of-office reply.'
  chosen:   Thanks for your email! I'm out of office until Monday and will reply a...
  rejected: I am currently unavailable....



Notice both examples are for the *same prompt* — DPO learns from the relative difference
between two responses to an identical question, not from an absolute "correct answer"
the way supervised fine-tuning does. In the first example, "chosen" isn't more factually
correct than "rejected" (both are true) — it's simply better suited to a 10-year-old,
which is exactly the kind of nuance that plain instruction-tuning data can't teach.

## 2. Where Preference Pairs Come From (and a Toy Generator)

In practice, preference pairs are built either from human comparisons or from a strong
LLM acting as a judge (the same technique used in the Evaluating LLMs notebook). Here's a
minimal, illustrative pattern for turning "judge picks a winner" into a training example.

In [2]:
def build_preference_example(prompt, response_a, response_b, winner):
    """winner should be 'a' or 'b' -- whichever response a human or judge preferred."""
    if winner == "a":
        chosen, rejected = response_a, response_b
    else:
        chosen, rejected = response_b, response_a
    return {"prompt": prompt, "chosen": chosen, "rejected": rejected}


# Simulating a judge's decision (in Topic 6's notebook, this comes from an actual LLM call)
example = build_preference_example(
    prompt="Suggest a name for a friendly robot vacuum.",
    response_a="Model X-4471B",
    response_b="Rolo",
    winner="b",   # the judge preferred the friendlier, more approachable name
)
print(example)

{'prompt': 'Suggest a name for a friendly robot vacuum.', 'chosen': 'Rolo', 'rejected': 'Model X-4471B'}


## 3. Building a Small Preference Dataset

For this notebook we'll use a handful of hand-written examples so the pipeline is fully
visible end to end. In a real project, you'd have hundreds to thousands of these, ideally
covering a wide range of prompts your model will actually see in production.

In [3]:
from datasets import Dataset

preference_data = [
    {
        "prompt": "Explain photosynthesis to a 10-year-old.",
        "chosen": "Plants use sunlight, water, and air to make their own food -- "
                   "like a tiny kitchen powered by the sun!",
        "rejected": "Photosynthesis is the process by which chlorophyll-containing "
                     "organisms convert electromagnetic radiation into chemical energy.",
    },
    {
        "prompt": "Write a short, friendly out-of-office reply.",
        "chosen": "Thanks for your email! I'm out of office until Monday and will "
                   "reply as soon as I'm back. For anything urgent, contact Sam.",
        "rejected": "I am currently unavailable.",
    },
    {
        "prompt": "Give a tip for staying focused while studying.",
        "chosen": "Try the Pomodoro technique: study in focused 25-minute bursts, "
                   "with a 5-minute break in between.",
        "rejected": "Focus harder and don't get distracted.",
    },
    {
        "prompt": "Suggest a name for a friendly robot vacuum.",
        "chosen": "Rolo",
        "rejected": "Model X-4471B",
    },
    # ... a real DPO dataset needs many more examples than this -- often thousands
]

dpo_dataset = Dataset.from_list(preference_data)
print(dpo_dataset)
print()
print(dpo_dataset[0])

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 4
})

{'prompt': 'Explain photosynthesis to a 10-year-old.', 'chosen': 'Plants use sunlight, water, and air to make their own food -- like a tiny kitchen powered by the sun!', 'rejected': 'Photosynthesis is the process by which chlorophyll-containing organisms convert electromagnetic radiation into chemical energy.'}


## 4. Load the SFT'd Model & Patch in DPO Support

We load our already fine-tuned checkpoint from the LoRA & QLoRA notebook (`./lora_model`)
rather than the raw base model — DPO is a refinement step *on top of* an instruction-tuned
model, not a replacement for that stage. `PatchDPOTrainer()` applies Unsloth's DPO-specific
speed and memory optimizations before we build the trainer.

In [ ]:
from unsloth import FastLanguageModel, PatchDPOTrainer

PatchDPOTrainer()   # must be called before creating the DPOTrainer, applies Unsloth's optimizations

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "lora_model",   # our SFT checkpoint saved at the end of the LoRA/QLoRA notebook
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# Re-attach a (fresh) LoRA adapter for the DPO stage itself
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha = 16,
    use_gradient_checkpointing = "unsloth",
)

print("SFT checkpoint loaded and re-wrapped with a LoRA adapter for DPO training.")

## 5. Configure & Run `DPOTrainer`

`DPOTrainer` from TRL implements the DPO loss from the slides directly — you don't have
to write the math yourself. The key hyperparameter is **`beta`**: it controls how strongly
training pushes the model toward the chosen response relative to the frozen reference
model (a copy of the checkpoint we just loaded, kept internally by the trainer).

- **Lower `beta`** (e.g. 0.1) → more willing to move away from the reference model.
- **Higher `beta`** (e.g. 0.5) → stays closer to the reference model's original behavior.

In [ ]:
from trl import DPOTrainer, DPOConfig

dpo_trainer = DPOTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dpo_dataset,
    args = DPOConfig(
        beta = 0.1,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 2,
        num_train_epochs = 3,       # tiny dataset, so a few epochs over it is reasonable
        learning_rate = 5e-6,        # DPO typically uses a smaller LR than SFT
        logging_steps = 1,
        optim = "adamw_8bit",
        seed = 3407,
        output_dir = "dpo_outputs",
        max_length = max_seq_length,
        max_prompt_length = 512,
    ),
)

dpo_trainer_stats = dpo_trainer.train()

## 6. Compare Outputs Before vs. After DPO

The real test of whether DPO worked: run the same prompts through the model before and
after training, and read the difference directly.

In [ ]:
FastLanguageModel.for_inference(model)

def generate_response(model, tokenizer, prompt, max_new_tokens=80):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, use_cache=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_prompt = "Write a short, friendly out-of-office reply."

print("AFTER DPO:")
print(generate_response(model, tokenizer, test_prompt))
print()
print("Compare this against the SFT-only model's response from the LoRA & QLoRA notebook --")
print("DPO should have nudged it toward the warmer, more detailed style from our 'chosen' examples.")

## 7. Inspecting the DPO Loss & Reward Margin

Beyond the raw loss, TRL's `DPOTrainer` also logs the **reward margin** — the gap between
how much the model favors the chosen response versus the rejected one. A healthy training
run shows this margin growing over time.

In [ ]:
import matplotlib.pyplot as plt

log_history = dpo_trainer.state.log_history
steps = [e["step"] for e in log_history if "loss" in e]
losses = [e["loss"] for e in log_history if "loss" in e]
margins = [e.get("rewards/margins", None) for e in log_history if "loss" in e]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(steps, losses, marker="o", markersize=3, color="tab:blue")
axes[0].set_title("DPO Loss")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].grid(alpha=0.3)

if any(m is not None for m in margins):
    axes[1].plot(steps, margins, marker="o", markersize=3, color="tab:green")
    axes[1].set_title("Reward Margin (chosen - rejected)")
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Margin")
    axes[1].axhline(0, color="gray", linewidth=0.8)
    axes[1].grid(alpha=0.3)
else:
    axes[1].text(0.5, 0.5, "Margin not logged\nat this logging_steps interval",
                  ha="center", va="center")

plt.tight_layout()
plt.show()

A **rising, positive reward margin** is the clearest sign DPO is working: it means the
model is increasingly more confident in the chosen response than the rejected one, for
the same prompt.

## 8. Save the DPO-tuned Model

Same export options as the SFT stage — save the adapter, merge to full precision, or
export to GGUF.

In [ ]:
model.save_pretrained("dpo_model")
tokenizer.save_pretrained("dpo_model")
print("Saved DPO-tuned adapter to ./dpo_model")

# Optional, same as the LoRA/QLoRA notebook:
# model.save_pretrained_merged("dpo_model_16bit", tokenizer, save_method="merged_16bit")
# model.push_to_hub("your-username/llama-3.1-8b-alpaca-dpo", tokenizer=tokenizer)

## Recap & Try It Yourself

You just:
- Saw exactly what a (prompt, chosen, rejected) preference triplet looks like.
- Built a small preference dataset by hand.
- Loaded an SFT checkpoint and patched in Unsloth's DPO optimizations.
- Trained with TRL's `DPOTrainer`, using `beta` to control how strongly to shift behavior.
- Compared outputs before and after DPO, and inspected the loss and reward margin.

**Things to try:**
1. Add 5-10 more preference pairs of your own to `preference_data` and re-run training.
2. Change `beta` from 0.1 to 0.5 and compare how much the model's outputs shift.
3. Feed the same test prompts through both the SFT-only model (`lora_model`) and the
   DPO model (`dpo_model`) and compare responses side by side.
4. Connect this to the Evaluating LLMs notebook: use the LLM-as-judge script there to
   score SFT-only vs. DPO outputs automatically, instead of reading them by eye.